# Predicting Student Dropout and Academic Success

Use student demographic, academic, and socioeconomic data to identify students at risk of dropping out, remaining enrolled, or succeeding academically.

**Project Question:** How can an institution identify students who may need support early enough to improve student outcomes?

[Project Dataset](https://archive.ics.uci.edu/dataset/697/predict+students+dropout+and+academic+success)

**Models kept:** Random Forest and XGBoost (best performers). Other classifiers, PCA/t-SNE/UMAP, and the stacking ensemble were removed during cleanup.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# For Kim:
import pandas as pd

# df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/iX class project/dropout_data.csv', sep=';')
# display(df)

In [ ]:
# For Hudson:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/data.csv', sep=';')
display(df)

In [ ]:
# For Jacari:
import pandas as pd

# df = pd.read_csv('/content/dropout_data.csv', sep=';')
# display(df)

## Data Cleaning

The dataset creators performed rigorous preprocessing (anomalies, outliers, missing values), so little cleaning is needed. We confirm below.

In [ ]:
# Confirm no missing values and review distributions
display(df.isnull().sum())
display(df.describe())

In [ ]:
print("Categorical columns:", df.select_dtypes(include='object').columns.tolist())
print("Numerical columns:", df.select_dtypes(include='number').columns.tolist())

## Data Exploration

Correlation of each numerical feature with dropout status.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

# Binary dropout flag for correlation analysis
df['Is_Dropout'] = (df['Target'] == 'Dropout').astype(int)

numerical_cols = df.select_dtypes(include='number').columns.tolist()
dropout_correlation = df[numerical_cols].corr()[['Is_Dropout']].sort_values(by='Is_Dropout', ascending=False)

plt.figure(figsize=(8, 15))
sns.heatmap(dropout_correlation, annot=True, cmap='coolwarm', fmt=".2f", cbar=True)
plt.title('Correlation of Numerical Features with Dropout Status')
plt.yticks(rotation=0)
plt.show()

## Feature Engineering

Engineer first-semester performance and financial-risk features.

In [ ]:
# 1. Approval rate (share of enrolled 1st-sem classes passed)
df["approval_rate_1st_sem"] = (
    df["Curricular units 1st sem (approved)"] /
    df["Curricular units 1st sem (enrolled)"]
).fillna(0)

# 2. Failed courses, 1st sem
df["failed_courses_1st_sem"] = (
    df["Curricular units 1st sem (enrolled)"] -
    df["Curricular units 1st sem (approved)"]
)

# 3. Average 1st-sem grade
df["avg_first_sem_grade"] = (
    df["Curricular units 1st sem (grade)"] /
    df["Curricular units 1st sem (approved)"]
).fillna(0)

# 4. Financial risk flag: debtor OR tuition not up to date
df["financial_risk"] = (
    (df["Debtor"] == 1) | (df["Tuition fees up to date"] == 0)
).astype(int)

new_features = ["approval_rate_1st_sem", "failed_courses_1st_sem", "avg_first_sem_grade", "financial_risk"]
print(df[new_features].head())

## Data Encoding and Final Cleaning

Drop redundant columns and map the `Target` to numbers (Dropout=0, Enrolled=1, Graduate=2).

In [ ]:
# Drop low-value columns and the EDA-only Is_Dropout flag
columns_to_drop = ['Application mode', 'Nacionality', 'Application order', 'Is_Dropout']
existing_cols_to_drop = [col for col in columns_to_drop if col in df.columns]
df_prepared = df.drop(columns=existing_cols_to_drop)

target_mapping = {'Dropout': 0, 'Enrolled': 1, 'Graduate': 2}
df_prepared['Target'] = df_prepared['Target'].map(target_mapping)

print("Columns removed:", existing_cols_to_drop)
print("Target distribution:\n", df_prepared['Target'].value_counts())
display(df_prepared.head())

## Feature Scaling

Standardize features with `StandardScaler` (needed for clustering).

In [ ]:
from sklearn.preprocessing import StandardScaler

X = df_prepared.drop(columns=['Target'])
y = df_prepared['Target']

scaler = StandardScaler()
X_scaled_array = scaler.fit_transform(X)
df_scaled = pd.DataFrame(X_scaled_array, columns=X.columns)

print("Scaled feature shape:", df_scaled.shape)
display(df_scaled.head())

## K-Means Clustering (Feature Creation)

Group students by similarity and add the cluster label as a synthetic feature that captures non-linear structure.

In [ ]:
from sklearn.cluster import KMeans

inertia = []
k_range = range(1, 11)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(df_scaled)
    inertia.append(km.inertia_)

plt.figure(figsize=(10, 6))
plt.plot(k_range, inertia, marker='o', linestyle='--')
plt.title('Elbow Method for Optimal Clusters')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.xticks(k_range)
plt.grid(True)
plt.show()

In [ ]:
# k=5 chosen from elbow plot + project context
k_optimal = 5
kmeans = KMeans(n_clusters=k_optimal, random_state=42, n_init=10)
df_prepared['Cluster'] = kmeans.fit_predict(df_scaled)

print(f'Added Cluster feature with {k_optimal} clusters.')
print(df_prepared['Cluster'].value_counts())

## Recursive Feature Elimination (RFE)

Use RFECV with an XGBoost estimator to keep only the most predictive feature subset.

In [ ]:
import numpy as np
from xgboost import XGBClassifier
from sklearn.feature_selection import RFECV
from sklearn.model_selection import StratifiedKFold

X = df_prepared.drop(columns=['Target'])
y = df_prepared['Target']

estimator = XGBClassifier(
    objective='multi:softmax',
    num_class=len(y.unique()),
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=42
)

rfecv = RFECV(
    estimator=estimator,
    step=1,
    cv=StratifiedKFold(5),
    scoring='accuracy',
    min_features_to_select=1,
    n_jobs=-1
)
rfecv.fit(X, y)

print(f"Optimal number of features: {rfecv.n_features_}")
selected_features = X.columns[rfecv.support_]
print("Selected features:", selected_features.tolist())

plt.figure(figsize=(10, 6))
plt.xlabel("Number of features selected")
plt.ylabel("Cross validation score (accuracy)")
plt.plot(range(1, len(rfecv.cv_results_['mean_test_score']) + 1), rfecv.cv_results_['mean_test_score'])
plt.title('RFECV: Optimal number of features by accuracy')
plt.grid(True)
plt.show()

X_rfe = X[selected_features].copy()
print("Shape after RFE:", X_rfe.shape)

## Train-Test Split

Stratified 80/20 split to preserve class balance.

In [ ]:
from sklearn.model_selection import train_test_split

X_final = X_rfe
y_final = y

X_train, X_test, y_train, y_test = train_test_split(
    X_final, y_final, test_size=0.2, random_state=42, stratify=y_final
)

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

## Model Training: Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

print("Random Forest Performance:")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("\nClassification Report:\n", classification_report(y_test, y_pred_rf))

## Model Training: XGBoost

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score

xgb_model = XGBClassifier(
    objective='multi:softmax',
    num_class=len(y_train.unique()),
    eval_metric='mlogloss',
    random_state=42,
    use_label_encoder=False
)
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)

print("XGBoost Performance:")
print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("\nClassification Report:\n", classification_report(y_test, y_pred_xgb))

## Model Comparison: Random Forest vs XGBoost

In [ ]:
from sklearn.metrics import f1_score

model_predictions = {
    "Random Forest": y_pred_rf,
    "XGBoost": y_pred_xgb,
}

model_performance = []
for name, preds in model_predictions.items():
    model_performance.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, preds),
        'F1-Score (Weighted)': f1_score(y_test, preds, average='weighted')
    })

performance_df = pd.DataFrame(model_performance)
performance_melted = performance_df.melt(id_vars='Model', var_name='Metric', value_name='Score')

plt.figure(figsize=(8, 6))
sns.barplot(x='Model', y='Score', hue='Metric', data=performance_melted, palette='viridis')
plt.title('Random Forest vs XGBoost')
plt.ylim(0, 1)
plt.legend(title='Metric')
plt.tight_layout()
plt.show()

display(performance_df.sort_values(by='Accuracy', ascending=False))

## Model Evaluation: ROC Curves and AUC

One-vs-rest ROC curves for Random Forest and XGBoost across the three classes.

In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

y_pred_proba_rf = rf_model.predict_proba(X_test)
y_pred_proba_xgb = xgb_model.predict_proba(X_test)

y_test_binarized = label_binarize(y_test, classes=[0, 1, 2])
n_classes = y_test_binarized.shape[1]

model_probs = {'Random Forest': y_pred_proba_rf, 'XGBoost': y_pred_proba_xgb}
roc_data = {name: {'fpr': {}, 'tpr': {}, 'auc': {}} for name in model_probs}

for name, proba in model_probs.items():
    for i in range(n_classes):
        fpr, tpr, _ = roc_curve(y_test_binarized[:, i], proba[:, i])
        roc_data[name]['fpr'][i] = fpr
        roc_data[name]['tpr'][i] = tpr
        roc_data[name]['auc'][i] = auc(fpr, tpr)

class_labels = ['Dropout (0)', 'Enrolled (1)', 'Graduate (2)']
colors = ['blue', 'red', 'green']
linestyles = ['-', '--']

plt.figure(figsize=(15, 7))
for i in range(n_classes):
    plt.subplot(1, n_classes, i + 1)
    for j, (name, data) in enumerate(roc_data.items()):
        plt.plot(data['fpr'][i], data['tpr'][i], color=colors[i], linestyle=linestyles[j],
                 label=f'{name} (AUC = {data["auc"][i]:.2f})')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlim([0.0, 1.0]); plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
    plt.title(f'ROC: {class_labels[i]} (OvR)')
    plt.legend(loc='lower right'); plt.grid(True)
plt.tight_layout()
plt.show()

## Hyperparameter Tuning: Random Forest

In [ ]:
from sklearn.model_selection import GridSearchCV
import time

param_grid_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'random_state': [42]
}

grid_search_rf = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid_rf,
    cv=3, scoring='accuracy', n_jobs=-1, verbose=2
)

print("Tuning Random Forest...")
start = time.time()
grid_search_rf.fit(X_train, y_train)
print(f"Done in {time.time() - start:.2f}s.")

best_params_rf = grid_search_rf.best_params_
print(f"\nBest params: {best_params_rf}")
print(f"Best CV accuracy: {grid_search_rf.best_score_:.4f}")

In [ ]:
tuned_rf_model = RandomForestClassifier(**best_params_rf)
tuned_rf_model.fit(X_train, y_train)
y_pred_tuned_rf = tuned_rf_model.predict(X_test)

print("Tuned Random Forest Performance:")
print("Accuracy:", accuracy_score(y_test, y_pred_tuned_rf))
print("\nClassification Report:\n", classification_report(y_test, y_pred_tuned_rf))

performance_df = pd.concat([performance_df, pd.DataFrame([{
    'Model': 'Tuned Random Forest',
    'Accuracy': accuracy_score(y_test, y_pred_tuned_rf),
    'F1-Score (Weighted)': f1_score(y_test, y_pred_tuned_rf, average='weighted')
}])], ignore_index=True)
display(performance_df.sort_values(by='Accuracy', ascending=False))

## Hyperparameter Tuning: XGBoost

In [ ]:
param_grid_xgb = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'random_state': [42]
}

grid_search_xgb = GridSearchCV(
    estimator=XGBClassifier(
        objective='multi:softmax',
        num_class=len(y_train.unique()),
        eval_metric='mlogloss',
        use_label_encoder=False,
        random_state=42
    ),
    param_grid=param_grid_xgb,
    cv=3, scoring='accuracy', n_jobs=-1, verbose=2
)

print("Tuning XGBoost...")
start = time.time()
grid_search_xgb.fit(X_train, y_train)
print(f"Done in {time.time() - start:.2f}s.")

best_params_xgb = grid_search_xgb.best_params_
print(f"\nBest params: {best_params_xgb}")
print(f"Best CV accuracy: {grid_search_xgb.best_score_:.4f}")

In [ ]:
tuned_xgb_model = XGBClassifier(**best_params_xgb)
tuned_xgb_model.fit(X_train, y_train)
y_pred_tuned_xgb = tuned_xgb_model.predict(X_test)

print("Tuned XGBoost Performance:")
print("Accuracy:", accuracy_score(y_test, y_pred_tuned_xgb))
print("\nClassification Report:\n", classification_report(y_test, y_pred_tuned_xgb))

performance_df = pd.concat([performance_df, pd.DataFrame([{
    'Model': 'Tuned XGBoost',
    'Accuracy': accuracy_score(y_test, y_pred_tuned_xgb),
    'F1-Score (Weighted)': f1_score(y_test, y_pred_tuned_xgb, average='weighted')
}])], ignore_index=True)
display(performance_df.sort_values(by='Accuracy', ascending=False))

## Feature Importance: Random Forest

Which features most drive the model's predictions.

In [ ]:
feature_importances = rf_model.feature_importances_
features_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': feature_importances
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Feature', data=features_df)
plt.title('Random Forest Feature Importances')
plt.tight_layout()
plt.show()

print("Top 10 features:")
display(features_df.head(10))

## Confusion Matrix: Random Forest

In [ ]:
from sklearn.metrics import confusion_matrix

cm_rf = confusion_matrix(y_test, y_pred_rf)
class_names_cm = ['Dropout', 'Enrolled', 'Graduate']

plt.figure(figsize=(8, 6))
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names_cm, yticklabels=class_names_cm)
plt.title('Confusion Matrix: Random Forest')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

### Top 5 Most Important Features Influencing Dropout

In [ ]:
top_5_features = features_df.head(5)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=top_5_features, palette='viridis')
plt.title('Top 5 Features Influencing Dropout (Random Forest)', fontsize=14)
plt.tight_layout()
plt.show()

display(top_5_features)

## Further Feature Engineering: Performance Trends

Add cross-semester trend and overall-success features, then re-split and retrain to measure their impact.

In [ ]:
# Trend and overall-success features (operate on RFE-selected X_final)
X_final['approved_units_diff'] = X_final['Curricular units 2nd sem (approved)'] - X_final['Curricular units 1st sem (approved)']

total_enrolled = X_final['Curricular units 1st sem (enrolled)'] + X_final['Curricular units 2nd sem (enrolled)']
total_approved = X_final['Curricular units 1st sem (approved)'] + X_final['Curricular units 2nd sem (approved)']
X_final['overall_approval_rate'] = (total_approved / total_enrolled).fillna(0)

sum_grades_1st_sem = X_final['avg_first_sem_grade'] * X_final['Curricular units 1st sem (approved)']
sum_grades_2nd_sem = X_final['Curricular units 2nd sem (grade)'] * X_final['Curricular units 2nd sem (approved)']
total_approved_for_grade_avg = X_final['Curricular units 1st sem (approved)'] + X_final['Curricular units 2nd sem (approved)']
X_final['overall_grade_avg'] = ((sum_grades_1st_sem + sum_grades_2nd_sem) / total_approved_for_grade_avg).fillna(0)

display(X_final[['approved_units_diff', 'overall_approval_rate', 'overall_grade_avg']].head())

In [ ]:
# Additional semester-level features (pulled from df_prepared, which has all columns)
X_final["second_sem_pass_rate"] = (
    df_prepared["Curricular units 2nd sem (approved)"] / df_prepared["Curricular units 2nd sem (enrolled)"]
).fillna(0)

X_final["grade_improvement"] = (
    df_prepared["Curricular units 2nd sem (grade)"] - df_prepared["Curricular units 1st sem (grade)"]
).fillna(0)

X_final["failures_2nd_sem"] = (
    df_prepared["Curricular units 2nd sem (enrolled)"] - df_prepared["Curricular units 2nd sem (approved)"]
)

X_final["missed_eval_ratio_1st"] = (
    df_prepared["Curricular units 1st sem (without evaluations)"] / df_prepared["Curricular units 1st sem (enrolled)"]
).fillna(0)

X_final["missed_eval_ratio_2nd"] = (
    df_prepared["Curricular units 2nd sem (without evaluations)"] / df_prepared["Curricular units 2nd sem (enrolled)"]
).fillna(0)

X_final["financial_risk_score"] = df_prepared["Debtor"] + (1 - df_prepared["Tuition fees up to date"])

X_final["parent_education_avg"] = (
    (df_prepared["Mother's qualification"] + df_prepared["Father's qualification"]) / 2
).fillna(0)

print("New features added. X_final shape:", X_final.shape)
display(X_final[[
    'second_sem_pass_rate', 'grade_improvement', 'failures_2nd_sem',
    'missed_eval_ratio_1st', 'missed_eval_ratio_2nd', 'financial_risk_score', 'parent_education_avg'
]].head())

In [ ]:
# Re-split with the expanded feature set
X_train, X_test, y_train, y_test = train_test_split(
    X_final, y_final, test_size=0.2, random_state=42, stratify=y_final
)
print("Re-split done. X_train:", X_train.shape, "X_test:", X_test.shape)

In [ ]:
# Retrain Random Forest with tuned params on the expanded features
retrained_rf_model = RandomForestClassifier(
    max_depth=10, min_samples_leaf=2, min_samples_split=10,
    n_estimators=100, random_state=42
)
retrained_rf_model.fit(X_train, y_train)
y_pred_retrained_rf = retrained_rf_model.predict(X_test)

print("Retrained Random Forest (with new features):")
print("Accuracy:", accuracy_score(y_test, y_pred_retrained_rf))
print("\nClassification Report:\n", classification_report(y_test, y_pred_retrained_rf))

In [ ]:
# Feature importance after expanded feature set
features_df_retrained = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': retrained_rf_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Feature', data=features_df_retrained.head(10), palette='viridis')
plt.title('Top 10 Features (Retrained Random Forest with New Features)', fontsize=14)
plt.tight_layout()
plt.show()

display(features_df_retrained.head(10))

## Interpret Predictions with SHAP

Use SHAP to explain individual student predictions and per-class feature impact for the tuned Random Forest.

In [ ]:
import sys
!{sys.executable} -m pip install shap
import shap

# TreeExplainer on the tuned Random Forest
explainer = shap.TreeExplainer(tuned_rf_model, feature_names=X_train.columns)
print("SHAP explainer initialized.")

In [ ]:
# Sample a few students to explain individually
sample_X_test = X_test.sample(n=5, random_state=42)
shap_values = explainer.shap_values(sample_X_test)
print(f"Computed SHAP values for {sample_X_test.shape[0]} students.")
display(sample_X_test.head())

In [ ]:
# Waterfall plot for one student's predicted class
class_names = ['Dropout', 'Enrolled', 'Graduate']
instance_idx = 0
sample_instance = sample_X_test.iloc[[instance_idx]]
predicted_class = tuned_rf_model.predict(sample_instance)[0]
print(f"Student {sample_instance.index[0]} predicted as '{class_names[predicted_class]}'")

# TreeExplainer multi-class output: shape (n_samples, n_features, n_classes)
shap_values_for_instance = shap_values[instance_idx, :, predicted_class]

explanation = shap.Explanation(
    values=shap_values_for_instance,
    base_values=explainer.expected_value[predicted_class],
    data=sample_X_test.iloc[instance_idx].values,
    feature_names=X_train.columns.tolist()
)

shap.waterfall_plot(explanation, max_display=10, show=False)
plt.title(f'SHAP Waterfall: Student {sample_instance.index[0]} (Predicted: {class_names[predicted_class]})')
plt.tight_layout()
plt.show()

In [ ]:
# Global feature impact across classes
shap.summary_plot(shap_values, sample_X_test, plot_type="violin", class_names=class_names)

In [ ]:
# Per-class SHAP feature importance for the 'Enrolled' class (class 1)
shap_values_retrained = explainer.shap_values(X_test)
class_to_analyze = 1
shap_values_enrolled = shap_values_retrained[:, :, class_to_analyze]
mean_abs_shap_enrolled = np.abs(shap_values_enrolled).mean(axis=0)

feature_importance_df_enrolled = pd.DataFrame({
    'Feature': X_test.columns,
    'Importance': mean_abs_shap_enrolled
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Feature', data=feature_importance_df_enrolled.head(10), palette='viridis')
plt.title(f'Top 10 Features Influencing "{class_names[class_to_analyze]}" (Mean |SHAP|)', fontsize=14)
plt.tight_layout()
plt.show()

display(feature_importance_df_enrolled.head(10))

## Conclusion: Key Drivers and Action Plan

**Top features influencing student success:**

- **Overall Approval Rate** — units approved across both semesters; the single strongest signal of consistent academic performance.
- **Second Semester Pass Rate** — proportion of 2nd-sem units passed; strong indicator of continued success.
- **Second Semester Failures** — count of 2nd-sem units failed; significant predictor of negative outcomes.
- **First Semester Approval Rate** — early-warning sign when low.
- **Curricular Units 2nd Sem (Approved)** — raw count of completed 2nd-sem units.

**Three plans of action:**

1. **Early Warning & Targeted Intervention** — monitor `approval_rate_1st_sem` and early failures by mid-first-semester; flag low performers for mandatory tutoring, counseling, and personalized success plans.
2. **Second-Semester Reinforcement & Retention** — build on first-semester support with time-management, study-skill, and stress workshops; tie `second_sem_pass_rate` and `failures_2nd_sem` to one-on-one coaching and peer/faculty mentorship.
3. **Holistic Well-being & Resource Integration** — a multi-department task force (advising, mental health, financial aid, career services) to address non-academic barriers for students flagged by the early-warning system.